# Build sporting dataset

This notebook transforms the validated sporting staging data into the analytical sporting dataset used for the financial + sporting merge.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv('../data/staging/sporting_data_staging.csv')
sporting = df.copy()

## Derive sporting metrics

The analytical metrics are derived from the same competition stage used in staging.

In [3]:
sporting['points_per_game'] = (sporting['points'] / sporting['matches_played'])
sporting['goal_difference_per_game'] = (sporting['goals_for'] - sporting['goals_against']) / sporting['matches_played']
sporting['relative_league_position'] = (sporting['league_size'] - sporting['league_position']) / (sporting['league_size'] - 1)

## Check derived metrics

In [8]:
derived_variables = ['points_per_game', 'goal_difference_per_game', 'relative_league_position']

sporting[derived_variables].isna().sum()


points_per_game             0
goal_difference_per_game    0
relative_league_position    0
dtype: int64

In [9]:
sporting[derived_variables].describe()

,points_per_game,goal_difference_per_game,relative_league_position
count,1256.000000,1256.000000,1256.000000
mean,1.355940,0.000004,0.500000
std,0.464325,0.710848,0.328407
min,0.000000,-4.333333,0.000000
25%,1.078947,-0.333333,0.222222
50%,1.357143,0.026316,0.500000
75%,1.666667,0.421053,0.777778
max,3.000000,3.166667,1.000000


In [12]:
assert (sporting['matches_played'] > 0).all()
assert (sporting['league_size'] > 1).all()
assert sporting[derived_variables].notna().all().all()
assert np.isfinite(sporting[derived_variables].to_numpy()).all()

assert sporting['points_per_game'].between(0, 3).all()
assert sporting['relative_league_position'].between(0, 1).all()

## Check merge keys

The sporting dataset must have at most one national-league record per club and season before merging with the financial dataset.

In [13]:
duplicates = sporting[sporting.duplicated(subset=['club', 'season'], keep=False)]
duplicates

,club,season,division,stage,group,matches_played,points,goals_for,goals_against,league_position,league_size,promoted,relegated,table_source,table_source_url,outcome_source,notes,points_per_game,goal_difference_per_game,relative_league_position


In [14]:
assert duplicates.empty
assert set(sporting['division'].unique()).issubset({1, 2, 3, 4})

## Export dataset

Keep the validated staging fields for auditability and append the three derived analytical metrics.

In [16]:
output_path = Path('../data/processed/sporting_data.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

sporting.to_csv(output_path, index=False)